# Project Overview

# Architecture

# Configuration & Setup

In [14]:
from pathlib import Path
import pandas as pd

from pyspark.sql import SparkSession
from financial_lakehouse.pipeline import run_pipeline

from financial_lakehouse.config import BRONZE_DIR
from financial_lakehouse.config import SILVER_DIR
from financial_lakehouse.config import GOLD_DIR

from financial_lakehouse.config import (
    CLASSIFICATION_DIR,
    TIME_SERIES_DIR,
)

from financial_lakehouse.config import (
    DEFAULT_END_DATE,
    DEFAULT_START_DATE,
)


In [2]:
spark = (
    SparkSession.builder
    .appName("Financial Lakehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 14:44:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Run Lakehouse Pipeline

In [ ]:

run_pipeline(
    spark=spark,
    tickers=["AAPL", "JPM", "XOM"],
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
)

Starting financial market data lakehouse pipeline.
Downloaded AAPL: 2,514 rows
Downloaded JPM: 2,514 rows
Downloaded XOM: 2,514 rows


Pipeline completed successfully.


# Inspect Bronze

In [ ]:
bronze_files = list(
    Path("../data/lakehouse/bronze/ticker=AAPL").rglob("*.parquet")
)

print(f"Partitions: {len(bronze_files)}")
print(bronze_files[:3])

sample_bz = pd.read_parquet(bronze_files[0])
sample_bz.head()

Partitions: 120
[PosixPath('../data/lakehouse/bronze/ticker=AAPL/year=2020/month=03/data.parquet'), PosixPath('../data/lakehouse/bronze/ticker=AAPL/year=2020/month=04/data.parquet'), PosixPath('../data/lakehouse/bronze/ticker=AAPL/year=2020/month=05/data.parquet')]


,Date,Adj Close,Close,High,Low,Open,Volume,download_timestamp
0,2020-03-02,72.071609,74.702499,75.360001,69.430000,70.570000,341397200,2026-09-22 18:44:25.693787+00:00
1,2020-03-03,69.782677,72.330002,76.000000,71.449997,75.917503,319475600,2026-09-22 18:44:25.693787+00:00
2,2020-03-04,73.019508,75.684998,75.849998,73.282501,74.110001,219178400,2026-09-22 18:44:25.693787+00:00
3,2020-03-05,70.650970,73.230003,74.887497,72.852501,73.879997,187572800,2026-09-22 18:44:25.693787+00:00
4,2020-03-06,69.712723,72.257500,72.705002,70.307503,70.500000,226176800,2026-09-22 18:44:25.693787+00:00


In [ ]:
bronze_df = spark.read.parquet(str(BRONZE_DIR))

bronze_df.printSchema()
print(f"Bronze rows: {bronze_df.count():,}")

# Inspect Silver

In [7]:
silver_files = list(
    SILVER_DIR.glob("ticker=AAPL/**/*.parquet")
)

print(f"Partitions: {len(silver_files)}")
print(silver_files[:3])

sample_sl = pd.read_parquet(silver_files[0])
sample_sl.head()

Partitions: 120
[PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/silver/ticker=AAPL/year=2020/month=7/part-00000-a74e36a6-8780-4fd2-864f-be78436121e4.c000.snappy.parquet'), PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/silver/ticker=AAPL/year=2020/month=9/part-00000-a74e36a6-8780-4fd2-864f-be78436121e4.c000.snappy.parquet'), PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/silver/ticker=AAPL/year=2020/month=8/part-00000-a74e36a6-8780-4fd2-864f-be78436121e4.c000.snappy.parquet')]


,Date,Adj Close,Open,High,Low,Close,Volume,download_timestamp
0,2020-07-01,88.059418,91.279999,91.839996,90.977501,91.027496,110737200,2026-09-22 18:44:25.693787
1,2020-07-29,91.941071,93.750000,95.230003,93.712502,95.040001,90329200,2026-09-22 18:44:25.693787
2,2020-07-09,92.630371,96.262497,96.317497,94.672501,95.752502,125642800,2026-09-22 18:44:25.693787
3,2020-07-22,94.100777,96.692497,97.974998,96.602501,97.272499,89001600,2026-09-22 18:44:25.693787
4,2020-07-17,93.186615,96.987503,97.147499,95.839996,96.327499,92186800,2026-09-22 18:44:25.693787


In [8]:
silver_check = spark.read.parquet(str(SILVER_DIR))

print(f"Silver rows: {silver_check.count():,}")

silver_check.printSchema()
silver_check.show(5)

Silver rows: 7,542
root
 |-- Date: date (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- download_timestamp: timestamp (nullable = true)
 |-- ticker: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+----------+------------------+-----------------+------------------+------------------+------------------+---------+--------------------+------+----+-----+
|      Date|         Adj Close|             Open|              High|               Low|             Close|   Volume|  download_timestamp|ticker|year|month|
+----------+------------------+-----------------+------------------+------------------+------------------+---------+--------------------+------+----+-----+
|2018-10-17|  52.2535514831543|55.57500076293945| 55.65999984741211|54.834999084472656

# Inspect Gold

In [11]:
gold_files = list(
    GOLD_DIR.glob("ticker=AAPL/**/*.parquet")
)

print(f"Partitions: {len(gold_files)}")
print(gold_files[:3])

sample_gd = pd.read_parquet(gold_files[0])
sample_gd.head()

Partitions: 120
[PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/gold/ticker=AAPL/year=2020/month=7/part-00000-bf075ad2-6405-4321-b5a3-e54264c58cba.c000.snappy.parquet'), PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/gold/ticker=AAPL/year=2020/month=9/part-00000-bf075ad2-6405-4321-b5a3-e54264c58cba.c000.snappy.parquet'), PosixPath('/Users/rnitto/Projects/portfolio/financial-data-lakehouse/data/lakehouse/gold/ticker=AAPL/year=2020/month=8/part-00000-bf075ad2-6405-4321-b5a3-e54264c58cba.c000.snappy.parquet')]


,Date,Adj Close,Open,High,Low,Close,Volume,download_timestamp,daily_return,log_return,...,price_vs_ma5,price_vs_ma20,price_vs_ma50,volatility_5,volatility_20,volatility_50,avg_volume_20,momentum_20,next_close,next_day_up
0,2020-07-01,88.059418,91.279999,91.839996,90.977501,91.027496,110737200,2026-09-22 18:44:25.693787,-0.001891,-0.001893,...,1.006296,1.040085,1.127349,0.020589,0.020941,0.016783,154059380.0,0.119925,91.027496,0
1,2020-07-02,88.059418,91.962502,92.617500,90.910004,91.027496,114041600,2026-09-22 18:44:25.693787,0.000000,0.000000,...,1.006702,1.033914,1.121239,0.019673,0.020714,0.016485,155383440.0,0.129654,93.462502,1
2,2020-07-06,90.415024,92.500000,93.945000,92.467499,93.462502,118655600,2026-09-22 18:44:25.693787,0.026750,0.026399,...,1.022202,1.055227,1.144268,0.013106,0.020620,0.016691,154453700.0,0.127753,93.172501,0
3,2020-07-07,90.134499,93.852501,94.654999,93.057503,93.172501,112424400,2026-09-22 18:44:25.693787,-0.003103,-0.003108,...,1.012987,1.046161,1.134487,0.012422,0.020726,0.016418,155292200.0,0.117645,95.342499,1
4,2020-07-08,92.233719,94.180000,95.375000,94.089996,95.342499,117092000,2026-09-22 18:44:25.693787,0.023290,0.023023,...,1.027326,1.064939,1.154010,0.014709,0.020261,0.016589,153761180.0,0.108666,95.752502,1


In [13]:
gold_check = spark.read.parquet(str(GOLD_DIR))

print(f"Gold rows: {gold_check.count():,}")

gold_check.printSchema()
gold_check.show(5)

Gold rows: 7,542
root
 |-- Date: date (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- download_timestamp: timestamp (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- log_return: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- ma_5: double (nullable = true)
 |-- ma_20: double (nullable = true)
 |-- ma_50: double (nullable = true)
 |-- price_vs_ma5: double (nullable = true)
 |-- price_vs_ma20: double (nullable = true)
 |-- price_vs_ma50: double (nullable = true)
 |-- volatility_5: double (nullable = true)
 |-- volatility_20: double (nullable = true)
 |-- volatility_50: double (nullable = true)
 |-- avg_volume_20: double (nullable = true)
 |-- momentum_20: double (nullable = true)
 |-- next_close: double (nullable = true)
 |-- next_day_up: integer (nulla

# Validate Datasets

## Classification

In [ ]:
classification_df = spark.read.parquet(str(CLASSIFICATION_DIR))
print(f"Classification rows: {classification_df.count():,}")

In [18]:
print("\nClassification Dataset")
classification_df.printSchema()


Classification Dataset
root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- log_return: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- ma_5: double (nullable = true)
 |-- ma_20: double (nullable = true)
 |-- ma_50: double (nullable = true)
 |-- price_vs_ma5: double (nullable = true)
 |-- price_vs_ma20: double (nullable = true)
 |-- price_vs_ma50: double (nullable = true)
 |-- volatility_5: double (nullable = true)
 |-- volatility_20: double (nullable = true)
 |-- volatility_50: double (nullable = true)
 |-- avg_volume_20: double (nullable = true)
 |-- momentum_20: double (nullable = true)
 |-- next_day_up: integer (nullable = true)
 |-- ticker: string (nullable = true)



In [16]:
classification_df.groupBy("next_day_up").count().orderBy("next_day_up").show()

+-----------+-----+
|next_day_up|count|
+-----------+-----+
|          0| 3569|
|          1| 3910|
+-----------+-----+



In [17]:
classification_df.groupBy("ticker", "next_day_up").count().orderBy(
    "ticker",
    "next_day_up",
).show()

+------+-----------+-----+
|ticker|next_day_up|count|
+------+-----------+-----+
|  AAPL|          0| 1156|
|  AAPL|          1| 1337|
|   JPM|          0| 1190|
|   JPM|          1| 1303|
|   XOM|          0| 1223|
|   XOM|          1| 1270|
+------+-----------+-----+



## Time Series

In [19]:
time_series_df = spark.read.parquet(str(TIME_SERIES_DIR))
print(f"Time-series rows:    {time_series_df.count():,}")

Time-series rows:    7,482


In [20]:
print("Time-Series Dataset")
time_series_df.printSchema()

Time-Series Dataset
root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- log_return: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- ma_5: double (nullable = true)
 |-- ma_20: double (nullable = true)
 |-- ma_50: double (nullable = true)
 |-- price_vs_ma5: double (nullable = true)
 |-- price_vs_ma20: double (nullable = true)
 |-- price_vs_ma50: double (nullable = true)
 |-- volatility_5: double (nullable = true)
 |-- volatility_20: double (nullable = true)
 |-- volatility_50: double (nullable = true)
 |-- avg_volume_20: double (nullable = true)
 |-- momentum_20: double (nullable = true)
 |-- ticker: string (nullable = true)



# Key Observations

# Reproducibility & Next Steps